# Maintainability and Scalability Audit

- Date: 2026-03-13
- Scope: `lib/`, selected platform/workflow files, CI/test posture
- Focus: maintainability, scalability, coupling, testability, and long-term change cost

## Verdict

The codebase is functional and moving toward production, but it is **not yet structurally ready to scale cleanly**.

The main risks are:
- in-memory-only sync retry state,
- mixed localization systems,
- string-based contracts between layers,
- oversized multi-responsibility files,
- incomplete dependency injection and no real automated tests.

## Highest Priority Findings

### 1) Sync retry queue is in memory only
- Failed remote writes are stored only in a local in-process list.
- If the app is killed, restarted, or crashes, the retry queue is lost.
- This breaks offline-first guarantees and will become a reliability issue as usage grows.

References:
- `lib/services/sync.dart:29`
- `lib/services/sync.dart:58`
- `lib/services/sync.dart:132`

Impact:
- High operational risk
- Hard to reason about sync integrity
- User trust issue once data volume grows

### 2) Localization is split across two systems
- The app now uses both `GStrings` and generated `AppLocalizations`.
- That creates duplication and guarantees migration debt later.
- Every new screen or change now has two possible string paths.

References:
- `lib/core/constants.dart:119`
- `lib/main.dart:109`
- `lib/pages/login_page.dart:85`

Impact:
- Medium-to-high maintenance cost
- Inconsistent localization rollout
- More fragile content updates

### 3) Auth/UI coordination depends on magic strings
- Provider-to-UI flow uses string protocols like `VERIFY:$email`.
- Login also reroutes based on checking whether an error string contains specific wording.
- This is brittle and will break quietly when messages change.

References:
- `lib/services/providers.dart:225`
- `lib/pages/login_page.dart:64`
- `lib/pages/signup_page.dart:98`

Impact:
- High refactor risk
- Harder to test correctly
- Poor contract clarity between layers

## Structural Maintainability Findings

### 4) `providers.dart` is a feature bottleneck
- One file contains theme, auth, anchor, Daily 3, capture, habit, and responsibility state logic.
- It is already one of the largest files in the project.
- This will become a merge-conflict hotspot and slow down feature changes.

Reference:
- `lib/services/providers.dart`

Impact:
- Harder onboarding
- Poor domain separation
- Larger blast radius for small edits

### 5) Incomplete dependency injection
- Some flows still use global singletons directly from pages/router (`LocalDb.instance`, `Supabase.instance.client`).
- This weakens testability and makes behavior harder to substitute or isolate.

References:
- `lib/core/router.dart:40`
- `lib/core/router.dart:52`
- `lib/pages/anchor_page.dart:21`
- `lib/pages/login_page.dart:54`

Impact:
- Harder unit/widget testing
- More implicit coupling
- Slower architectural change later

### 6) Large multi-responsibility files are accumulating
- `constants.dart`, `theme.dart`, `profile_page.dart`, and `providers.dart` each carry too many concerns.
- This makes small changes more expensive than they should be.

Largest files observed:
- `lib/services/providers.dart` (~653 lines)
- `lib/pages/profile_page.dart` (~643 lines)
- `lib/core/constants.dart` (~464 lines)
- `lib/core/theme.dart` (~296 lines)

Impact:
- Reduced readability
- Slower review cycles
- Harder ownership boundaries

## Scalability Process Findings

### 7) No real automated test base yet
- CI had to be adjusted to skip tests because `test/` does not exist.
- This means architectural changes are currently protected mostly by manual verification.

Impact:
- Refactor risk grows quickly
- Slower confidence in shipping
- Regression detection remains manual

## Recommended Remediation Order

1. Persist the sync retry queue in local storage.
2. Replace string-based auth results with typed outcomes or sealed UI actions.
3. Split `providers.dart` into feature-level provider files.
4. Finish migration from `GStrings` to generated localization.
5. Add a minimal test suite around auth, sync queue, and daily lock logic.
6. Gradually remove direct singleton usage from UI/router and route all access through providers/services.

## Final Assessment

The app is making good progress operationally, but long-term maintainability is currently constrained by architectural shortcuts that were acceptable during early development and are now becoming scaling liabilities.

The most important next move is not a UI tweak or platform change. It is **stabilizing state contracts and persistence boundaries** so new features do not multiply technical debt.